# 02 — Descriptive Statistics

Phase 2 characterizes the distributions of every variable used in Phase 4 hypothesis testing.
Each section ends with a **Test Decision** block that Phase 4 inherits verbatim.

Scope (locked in Phase 1): delivered orders, Jan 2017 – Aug 2018 window.

## 1. Order value

Measured as `payment_value` — sum of payments per order, aggregated in master.

**Why this matters:** Phase 4 asks whether order value differs across product categories
(ANOVA-style question). That test wants roughly normal residuals and equal variances.
E-commerce order totals are almost always right-skewed; the job here is to quantify
*how* skewed and decide between log-transform + ANOVA vs Kruskal-Wallis.

In [23]:
# ── Phase 2: Descriptive Statistics ───────────────────────────────
# Setup, path anchor, and data load.

import os
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

pd.set_option('display.max_columns', 30)

# Anchor to project root so paths work regardless of kernel launch dir.
PROJECT_ROOT = Path(r'C:\Users\thinkpad\Desktop\olist-ecommerce-analysis')
assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)

FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'orders_master.parquet')
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded: 99,441 rows × 26 columns


## 1. Order value

Order value = `payment_value_total` (matches the brief's "sipariş fiyatı"; includes freight).

**Why this matters:** Phase 4 will test whether order value differs across product
categories (ANOVA-style question). That test wants roughly normal residuals.
E-commerce totals are typically right-skewed — this section quantifies how much
and decides between ANOVA on log vs Kruskal-Wallis.

In [24]:
window_start = pd.Timestamp('2017-01-01')
window_end   = pd.Timestamp('2018-08-31 23:59:59')

delivered = df[
    (df['order_status'] == 'delivered') &
    (df['order_purchase_timestamp'] >= window_start) &
    (df['order_purchase_timestamp'] <= window_end)
].copy()

print(f"Analytical sample: {len(delivered):,} delivered orders")
print(f"Window: {delivered['order_purchase_timestamp'].min().date()} → "
      f"{delivered['order_purchase_timestamp'].max().date()}")
print(f"Dropped from master: {len(df) - len(delivered):,} "
      f"({(len(df) - len(delivered))/len(df)*100:.1f}%)")

Analytical sample: 96,211 delivered orders
Window: 2017-01-05 → 2018-08-29
Dropped from master: 3,230 (3.2%)


In [25]:
pv = delivered['payment_value_total'].dropna()

summary = pd.Series({
    'n':        len(pv),
    'mean':     pv.mean(),
    'median':   pv.median(),
    'std':      pv.std(),
    'min':      pv.min(),
    'q25':      pv.quantile(0.25),
    'q75':      pv.quantile(0.75),
    'q95':      pv.quantile(0.95),
    'q99':      pv.quantile(0.99),
    'max':      pv.max(),
    'skew':     pv.skew(),
    'kurtosis': pv.kurtosis(),
}).round(2)

summary

n           96211.00
mean          159.81
median        105.28
std           218.88
min             9.59
q25            61.88
q75           176.26
q95           445.64
q99          1053.61
max         13664.08
skew            9.38
kurtosis      249.57
dtype: float64

In [26]:
# Does freight distort order value enough to confuse category comparisons in Phase 4?
freight_share = (delivered['freight_total'] / delivered['payment_value_total']).dropna()

print("Freight as share of order value:")
print(f"  mean:   {freight_share.mean()*100:5.1f}%")
print(f"  median: {freight_share.median()*100:5.1f}%")
print(f"  std:    {freight_share.std()*100:5.1f}%")
print(f"  p90:    {freight_share.quantile(0.9)*100:5.1f}%")

corr = delivered[['payment_value_total', 'items_price_total']].corr().iloc[0, 1]
print(f"\nCorrelation payment_value_total ↔ items_price_total: {corr:.4f}")

Freight as share of order value:
  mean:    20.9%
  median:  18.3%
  std:     12.6%
  p90:     38.3%

Correlation payment_value_total ↔ items_price_total: 0.9959


In [27]:
pv = delivered['payment_value_total'].dropna()
log_pv = np.log10(pv)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Order value (BRL) — raw', 'log10(order value)')
)
fig.add_trace(go.Histogram(x=pv,     nbinsx=100, marker_color='#4C78A8'), row=1, col=1)
fig.add_trace(go.Histogram(x=log_pv, nbinsx=60,  marker_color='#4C78A8'), row=1, col=2)

fig.update_layout(
    title='Order value — shape check (raw vs log10)',
    showlegend=False, height=420, template='plotly_white', bargap=0.02,
)
fig.update_xaxes(title_text='BRL',        row=1, col=1)
fig.update_xaxes(title_text='log10(BRL)', row=1, col=2)
fig.update_yaxes(title_text='count',      row=1, col=1)

fig.show()

fig_path = FIGURES_DIR / '01_order_value_distribution.png'
fig.write_image(fig_path, width=1200, height=500, scale=2)
print(f"Saved: {fig_path.relative_to(PROJECT_ROOT)}")

Saved: reports\figures\01_order_value_distribution.png


In [28]:
# On log scale — how much of the skew did the transform absorb?
stat, p = stats.normaltest(log_pv)
print("Log10(payment_value_total) — moments:")
print(f"  skew      = {log_pv.skew():+.3f}")
print(f"  kurtosis  = {log_pv.kurtosis():+.3f}")
print(f"  D'Agostino K²: stat={stat:,.1f}, p={p:.2e}")
print("  (With n≈97k, formal test always rejects — trust the moments, not p.)")
print()

# Business framing
gap_pct = (pv.mean() - pv.median()) / pv.median() * 100
top1 = pv[pv >= pv.quantile(0.99)]
print(f"Mean R$ {pv.mean():.2f}  |  Median R$ {pv.median():.2f}")
print(f"Mean is {gap_pct:.1f}% higher than median — right-skew signature.")
print(f"Top 1% (≥ R$ {pv.quantile(0.99):.2f}) = {len(top1):,} orders "
      f"but {top1.sum() / pv.sum() * 100:.1f}% of total GMV.")

Log10(payment_value_total) — moments:
  skew      = +0.522
  kurtosis  = +0.558
  D'Agostino K²: stat=4,651.8, p=0.00e+00
  (With n≈97k, formal test always rejects — trust the moments, not p.)

Mean R$ 159.81  |  Median R$ 105.28
Mean is 51.8% higher than median — right-skew signature.
Top 1% (≥ R$ 1053.61) = 963 orders but 10.4% of total GMV.


### Test Decision — Order value

| Finding | Value | Implication |
|---|---|---|
| n (delivered, windowed) | 96,211 | Ample power for any test |
| Raw skew / kurtosis | +9.38 / +249.6 | Extreme right-skew, heavy tail |
| Log10 skew / kurtosis | +0.52 / +0.56 | Approximately normal after transform |
| Mean vs median gap | 51.8% | Mean not interpretable as "typical" |
| Top-1% GMV share | 10.4% (963 orders) | Small bulk/B2B tail drives disproportionate revenue |
| Corr(payment_value, items_price) | 0.996 | Freight variance does not reorder observations |

**Phase 4 locked decision — category → order value comparison:**
- Primary test: **one-way ANOVA on `log10(payment_value_total)`**
- Equal-variance assumption check: Levene's test on log-values across categories
- Post-hoc (if ANOVA rejects): **Tukey HSD** on log-values
- Effect size: **eta-squared (η²)**
- Robustness: if Levene rejects homogeneity, fall back to Welch's ANOVA or Kruskal-Wallis

**Business "so what":**
- Mean order value (R$160) overstates typical customer spend by 52%. Median (R$105) is the honest center.
- 1% of orders (963 of them) drive 10.4% of GMV. Segmenting this tail is a separate exercise from describing the median customer — conflating them produces misleading dashboards.
- Freight averages 21% of order value but varies widely (std 13 pp). Shipping-subsidy decisions should be per-category, not platform-wide — queued for Phase 4 or future work.

## 2. Category distribution

The Phase 4 category → order value test (one-way ANOVA) needs groups with
adequate sample sizes. Olist's raw category column has dozens of values, many
with negligible counts.

**Decisions this section drives:**
1. Cutoff for Phase 4 inclusion — minimum n per category to participate in ANOVA
2. Whether a residual "other" bucket is analytically useful or adds noise
3. Identify the top categories that carry the business (Pareto check)

In [29]:
# Per-category counts + share of total orders
cat_counts = (
    delivered['category']
    .fillna('__missing__')
    .value_counts()
    .rename_axis('category')
    .to_frame('n_orders')
)
cat_counts['share_pct']     = cat_counts['n_orders'] / cat_counts['n_orders'].sum() * 100
cat_counts['cum_share_pct'] = cat_counts['share_pct'].cumsum()

print(f"Distinct category values (incl. missing): {len(cat_counts)}")
print(f"Missing category rows: {cat_counts.loc['__missing__', 'n_orders'] if '__missing__' in cat_counts.index else 0:,}")
print()
cat_counts.head(15)

Distinct category values (incl. missing): 72
Missing category rows: 1,376



,n_orders,share_pct,cum_share_pct
category,,,
bed_bath_table,9180,9.541529,9.541529
health_beauty,8576,8.913742,18.455270
sports_leisure,7476,7.770421,26.225691
computers_accessories,6500,6.755984,32.981676
furniture_decor,6140,6.381807,39.363482
housewares,5671,5.894336,45.257819
watches_gifts,5474,5.689578,50.947397
telephony,4066,4.226128,55.173525
auto,3785,3.934062,59.107586


In [30]:
# How thin is the tail?
real_cats = cat_counts.drop('__missing__', errors='ignore')

cutoffs = [30, 50, 100, 500, 1000]
print("Category participation at various minimum-n cutoffs:")
print(f"{'min_n':>7}  {'categories':>10}  {'orders_covered':>15}  {'coverage_pct':>13}")
for c in cutoffs:
    keep = real_cats[real_cats['n_orders'] >= c]
    n_orders = keep['n_orders'].sum()
    pct = n_orders / real_cats['n_orders'].sum() * 100
    print(f"{c:>7}  {len(keep):>10}  {n_orders:>15,}  {pct:>12.1f}%")

print()
print(f"Pareto check — how many categories to reach 80% of orders?")
n_for_80 = (real_cats['cum_share_pct'] <= 80).sum() + 1
print(f"  Top {n_for_80} categories cover "
      f"{real_cats['cum_share_pct'].iloc[n_for_80-1]:.1f}% of orders")

Category participation at various minimum-n cutoffs:
  min_n  categories   orders_covered   coverage_pct
     30          62           94,679          99.8%
     50          58           94,535          99.7%
    100          51           94,031          99.2%
    500          25           87,521          92.3%
   1000          21           84,836          89.5%

Pareto check — how many categories to reach 80% of orders?
  Top 16 categories cover 81.2% of orders


In [31]:
top_n = 20
top = real_cats.head(top_n).reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=top['category'], y=top['n_orders'],
           name='Orders', marker_color='#4C78A8'),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=top['category'], y=top['cum_share_pct'],
               name='Cumulative %', mode='lines+markers',
               line=dict(color='#E45756', width=2)),
    secondary_y=True,
)
fig.add_hline(y=80, line_dash='dot', line_color='#888',
              secondary_y=True, annotation_text='80%', annotation_position='right')

fig.update_layout(
    title=f'Top {top_n} categories — volume and cumulative share (Pareto view)',
    height=500, template='plotly_white', showlegend=True,
    xaxis_tickangle=-45,
)
fig.update_yaxes(title_text='Orders',          secondary_y=False)
fig.update_yaxes(title_text='Cumulative %', secondary_y=True, range=[0, 105])

fig.show()

fig_path = FIGURES_DIR / '02_category_pareto.png'
fig.write_image(fig_path, width=1400, height=600, scale=2)
print(f"Saved: {fig_path.relative_to(PROJECT_ROOT)}")

Saved: reports\figures\02_category_pareto.png


In [32]:
# Spell out exactly what gets cut at different thresholds — this is the evidence
# behind the Phase 4 locked cutoff.
print("Categories with fewer than 100 orders (candidates for exclusion):")
thin = real_cats[real_cats['n_orders'] < 100].sort_values('n_orders')
print(f"  count: {len(thin)}")
print(f"  total orders in these categories: {thin['n_orders'].sum():,} "
      f"({thin['n_orders'].sum() / real_cats['n_orders'].sum() * 100:.2f}% of all)")
print()
print("Thinnest 10:")
print(thin.head(10)[['n_orders', 'share_pct']])

Categories with fewer than 100 orders (candidates for exclusion):
  count: 20
  total orders in these categories: 804 (0.85% of all)

Thinnest 10:
                           n_orders  share_pct
category                                      
security_and_services             2   0.002079
fashion_childrens_clothes         7   0.007276
cds_dvds_musicals                12   0.012473
la_cuisine                       13   0.013512
arts_and_craftmanship            22   0.022866
home_comfort_2                   22   0.022866
diapers_and_hygiene              24   0.024945
fashion_sport                    25   0.025985
flowers                          29   0.030142
fashio_female_clothing           34   0.035339


### Test Decision — Category

| Finding | Value | Implication |
|---|---|---|
| Distinct categories (incl. missing) | 72 | Too many for clean ANOVA post-hoc |
| Missing category rows | 1,376 (1.4%) | Excluded from category test; noted as DQ caveat |
| Categories to reach 80% of orders | 16 | Moderate Pareto; no dominant category |
| Thin tail (n<100) | 20 categories, 0.85% of orders | Dropped for Phase 4 primary test |
| min_n=500 → 25 categories, 92.3% coverage | Primary cutoff | Balances readability and coverage |
| min_n=100 → 51 categories, 99.2% coverage | Robustness cutoff | Used as sensitivity check |

**Phase 4 locked decision — category → order value (one-way ANOVA on log10(payment_value_total)):**
- Primary sample: categories with n ≥ 500 (25 categories, 87,521 orders)
- Robustness sample: categories with n ≥ 100 (51 categories, 94,031 orders)
- Equal-variance assumption: Levene's test on log-values across categories
- Post-hoc: Tukey HSD on log-values (25-group scale is interpretable)
- Effect size: eta-squared (η²)
- If Levene rejects: Welch's ANOVA as primary, Games-Howell post-hoc

**Business "so what":**
- Olist's marketplace is broad, not concentrated — top category is 9.5% of orders, takes 16 categories to reach 80%. Category-specific campaigns reach smaller audiences than in concentrated verticals; cross-category bundles likely have more upside than in specialist retailers.
- The 20-category thin tail (0.85% of orders) includes structural anomalies
  (`security_and_services` n=2, `cds_dvds_musicals` n=12) rather than undersampled real categories — excluding them is a data-hygiene decision, not a scope limitation.